In [8]:
import pandas as pd
from functools import reduce

In [9]:
df = pd.read_csv('data_v1.csv', index_col=0)
dehumanizing = pd.read_csv('data_with_dehumanizing_labels.csv', index_col=0)

df = df[df['network'] != 'BBCNEWS']
dehumanizing = dehumanizing[dehumanizing['network'] != 'BBCNEWS']

df['network'] = df['network'].replace('MSNBCW', 'MSNOW')
dehumanizing['network'] = dehumanizing['network'].replace('MSNBCW', 'MSNOW')

df['date'] = pd.to_datetime(df['date'])
dehumanizing['date'] = pd.to_datetime(dehumanizing['date'])

df = df.set_index('date')

df = df.drop(columns=['duration(s)'])
dehumanizing = dehumanizing.drop(
    columns=['duration(s)', 'Unnamed: 0', 'show_tag'])

In [10]:
merged = pd.merge(df, dehumanizing, on=['date', 'network', 'llm_text', 'year'])

In [11]:
merged['hateful_label'] = [
    'hateful' if prob > .95 else 'not hateful' for prob in merged['speech_hateful_prob']]
merged['targeted_label'] = [
    'targeted' if pd.notna(val) else 'not targeted' for val in merged['country']]
merged['aggressive_label'] = [
    'aggressive' if prob > .7 else 'not aggressive' for prob in merged['speech_aggressive_prob']]

In [21]:
print("Number of unique Fox News shows: ",
      merged[merged['network'] == 'FOXNEWSW']['show_tag'].nunique())
print("Number of unique CNN shows: ",
      merged[merged['network'] == 'CNNW']['show_tag'].nunique())
print("Number of unique MSNOW shows: ",
      merged[merged['network'] == 'MSNOW']['show_tag'].nunique())

print("Number of Fox News chyrons: ",
      merged[merged['network'] == 'FOXNEWSW']['date'].nunique())
print("Number of CNN chyrons: ",
      merged[merged['network'] == 'CNNW']['date'].nunique())
print("Number of MSNOW chyrons: ",
      merged[merged['network'] == 'MSNOW']['date'].nunique())

Number of unique Fox News shows:  127
Number of unique CNN shows:  205
Number of unique MSNOW shows:  140
Number of Fox News chyrons:  38298
Number of CNN chyrons:  22069
Number of MSNOW chyrons:  16551


In [ ]:
merged.to_csv('data_v2.csv')

In [22]:
df = pd.read_csv('data_v2.csv', index_col=0)

In [23]:
counts_df = merged.drop(columns=['llm_text', 'show_tag',
                                 'speech_hateful_prob', 'speech_targeted_prob', 'speech_aggressive_prob',
                                 'emo_others_prob', 'emo_joy_prob', 'emo_sadness_prob',
                                 'emo_anger_prob', 'emo_surprise_prob', 'emo_disgust_prob',
                                 'emo_fear_prob', 'sent_pos_prob', 'sent_neg_prob',
                                 'sent_neu_prob', 'metaphor_category', 'country',
                                 'region', 'matched_frames', 'matched_terms', 'frame_count',
                                 'primary_frame', 'has_frame'])

In [24]:
cnn_counts = counts_df[counts_df['network'] == 'CNNW']
cnn_counts = cnn_counts.drop(columns=['network'])

fox_counts = counts_df[counts_df['network'] == 'FOXNEWSW']
fox_counts = fox_counts.drop(columns=['network'])

msnow_counts = counts_df[counts_df['network'] == 'MSNOW']
msnow_counts = msnow_counts.drop(columns=['network'])

In [27]:
def build_pct_table(df, year):
    labels = [
        'emo_label', 'sent_label', 'hateful_label', 'aggressive_label', 'targeted_label', 'is_dehumanizing']
    list_dfs = []
    filtered_df = df[df['year'] == year]
    for label in labels:
        label_counts = (filtered_df.groupby(
            ["network", label]).size().reset_index(name="count"))
        network_totals = (label_counts.groupby("network")[
            "count"].sum().reset_index(name="network_total"))
        pct_table = label_counts.merge(network_totals, on=["network"])
        pct_table["percent"] = (
            pct_table["count"] / pct_table["network_total"] * 100)
        pct_table = (pct_table.pivot(
            index="network", columns=label, values="percent").fillna(0).reset_index())
        list_dfs.append(pct_table)
    return list_dfs

In [28]:
dfs_2018 = build_pct_table(counts_df, 2018)
dfs_2019 = build_pct_table(counts_df, 2019)
dfs_2020 = build_pct_table(counts_df, 2020)
dfs_2021 = build_pct_table(counts_df, 2021)
dfs_2022 = build_pct_table(counts_df, 2022)
dfs_2023 = build_pct_table(counts_df, 2023)
dfs_2024 = build_pct_table(counts_df, 2024)
dfs_2025 = build_pct_table(counts_df, 2025)
dfs_2026 = build_pct_table(counts_df, 2026)

In [29]:
df_2018 = reduce(lambda left, right: pd.merge(left, right, on=['network'],
                                              how='outer'), dfs_2018)
df_2019 = reduce(lambda left, right: pd.merge(left, right, on=['network'],
                                              how='outer'), dfs_2019)
df_2020 = reduce(lambda left, right: pd.merge(left, right, on=['network'],
                                              how='outer'), dfs_2020)
df_2021 = reduce(lambda left, right: pd.merge(left, right, on=['network'],
                                              how='outer'), dfs_2021)
df_2022 = reduce(lambda left, right: pd.merge(left, right, on=['network'],
                                              how='outer'), dfs_2022)
df_2023 = reduce(lambda left, right: pd.merge(left, right, on=['network'],
                                              how='outer'), dfs_2023)
df_2024 = reduce(lambda left, right: pd.merge(left, right, on=['network'],
                                              how='outer'), dfs_2024)
df_2025 = reduce(lambda left, right: pd.merge(left, right, on=['network'],
                                              how='outer'), dfs_2025)
df_2026 = reduce(lambda left, right: pd.merge(left, right, on=['network'],
                                              how='outer'), dfs_2026)

df_2018['year'] = 2018
df_2019['year'] = 2019
df_2020['year'] = 2020
df_2021['year'] = 2021
df_2022['year'] = 2022
df_2023['year'] = 2023
df_2024['year'] = 2024
df_2025['year'] = 2025
df_2026['year'] = 2026

In [31]:
final_df = pd.concat([df_2018, df_2019, df_2020, df_2021,
                     df_2022, df_2023, df_2024, df_2025, df_2026])

In [32]:
final_df

,network,anger,disgust,fear,joy,others,sadness,surprise,NEG,NEU,POS,hateful,not hateful,aggressive,not aggressive,not targeted,targeted,False,True,year
0,CNNW,9.563133,28.236548,0.532765,0.745871,59.136921,1.758125,0.026638,48.508258,47.309536,4.182206,1.704848,98.295152,0.559403,99.440597,92.115077,7.884923,99.041023,0.958977,2018
1,FOXNEWSW,10.020331,23.729306,0.929422,0.682544,63.941330,0.682544,0.014522,38.687191,57.130410,4.182399,4.255010,95.744990,2.802788,97.197212,93.958757,6.041243,99.186756,0.813244,2018
2,MSNOW,8.426428,29.971274,0.223428,0.574529,59.304181,1.468241,0.031918,44.526013,50.686243,4.787743,1.563996,98.436004,0.638366,99.361634,95.244175,4.755825,99.202043,0.797957,2018
0,CNNW,6.786850,25.450689,1.519972,0.813008,63.449982,1.944150,0.035348,45.245670,50.265111,4.489219,1.343231,98.656769,0.600919,99.399081,91.622481,8.377519,99.045599,0.954401,2019
1,FOXNEWSW,7.853881,22.995434,1.351598,0.657534,66.356164,0.748858,0.036530,39.525114,56.219178,4.255708,3.068493,96.931507,1.990868,98.009132,91.141553,8.858447,98.940639,1.059361,2019
2,MSNOW,5.750408,27.161501,1.019576,0.244698,64.967374,0.815661,0.040783,42.414356,54.608483,2.977162,1.182708,98.817292,0.897227,99.102773,95.269168,4.730832,98.531811,1.468189,2019
0,CNNW,4.507042,13.802817,1.971831,1.126761,76.056338,2.535211,NaN,43.380282,52.394366,4.225352,0.000000,100.000000,0.000000,100.000000,85.070423,14.929577,96.056338,3.943662,2020
1,FOXNEWSW,7.093023,21.511628,1.046512,0.813953,69.186047,0.348837,NaN,35.348837,58.837209,5.813953,3.255814,96.744186,2.209302,97.790698,93.255814,6.744186,98.488372,1.511628,2020
2,MSNOW,5.432937,21.222411,1.018676,1.188455,70.458404,0.679117,NaN,39.728353,53.650255,6.621392,0.679117,99.320883,0.000000,100.000000,95.585739,4.414261,97.623090,2.376910,2020
0,CNNW,4.148279,19.064431,3.265666,0.617829,70.520741,2.383054,0.000000,44.218888,52.868491,2.912621,1.147396,98.852604,0.882613,99.117387,77.316858,22.683142,96.734334,3.265666,2021


In [286]:
final_df.to_csv('year_counts_hor.csv')